In [1]:
# Cell 1: Imports and feature calculation functions
import os
import joblib
import pandas as pd
import numpy as np
import math

def calculate_sortedness_sorting(arr):
    """Sortedness for sorting: strictly non-decreasing (ascending)."""
    if len(arr) < 2:
        return 1.0
    return 1.0 if all(arr[i] <= arr[i+1] for i in range(len(arr)-1)) else 0.0

def calculate_sortedness_searching(arr):
    """Sortedness for searching: ascending or descending allowed."""
    if len(arr) < 2:
        return 1.0
    non_decreasing = all(arr[i] <= arr[i+1] for i in range(len(arr)-1))
    non_increasing = all(arr[i] >= arr[i+1] for i in range(len(arr)-1))
    return 1.0 if non_decreasing or non_increasing else 0.0

def calculate_size(arr):
    return len(arr)

def calculate_uniqueness(arr):
    if len(arr) == 0:
        return 0.0
    return len(set(arr)) / len(arr)

def calculate_range(arr):
    if len(arr) == 0:
        return 0.0
    return max(arr) - min(arr)

def compute_distribution(arr, num_bins=10):
    hist, _ = np.histogram(arr, bins=num_bins)
    proportions = hist / len(arr)
    sparsity = np.sum(proportions > 0)
    if sparsity == num_bins:
        return 1  # uniform
    elif sparsity <= num_bins / 3:
        return 2  # clustered
    else:
        return 0  # random


In [2]:
# Cell 2: ML-based sorting prediction function
def predict_sorting_algorithm(arr, model_path='model.pkl'):
    size = calculate_size(arr)
    sortedness = calculate_sortedness_sorting(arr)  # strict ascending
    uniqueness = calculate_uniqueness(arr)
    range_val = calculate_range(arr)

    if not os.path.exists(model_path):
        print("Model file not found. Please train 'model.pkl' first.")
        return None

    clf = joblib.load(model_path)
    cols = ["size", "sortedness", "uniqueness", "range"]
    X = pd.DataFrame([[size, sortedness, uniqueness, range_val]], columns=cols)
    pred = clf.predict(X)[0]
    return pred


In [3]:
# Cell 3: Rule-based search algorithm selection
def select_search_algorithm(arr, num_queries):
    n = len(arr)
    if n == 0:
        return 'linear_search'

    sortedness_flag = 1 if calculate_sortedness_searching(arr) == 1.0 else 0
    distribution = compute_distribution(arr)

    small_n_threshold = 1000

    if sortedness_flag != 1:
        # Unsorted — consider sorting costs vs linear search
        sort_cost = n * math.log2(n) if n > 1 else 0
        linear_cost = num_queries * n
        binary_cost = math.log2(n) + sort_cost
        interp_cost = num_queries * (math.log2(math.log2(n)) if distribution == 1 else n) + sort_cost

        costs = {
            'linear_search': linear_cost,
            'sort_and_binary': binary_cost,
            'sort_and_interpolation': interp_cost
        }
        return min(costs, key=costs.get)

    else:
        # Sorted (ascending or descending)
        if n <= small_n_threshold:
            if distribution == 1:
                return 'interpolation_search'
            else:
                return 'binary_search'

        if distribution == 1:
            return 'interpolation_search'

        # For non-uniform, pick among jump, ternary, binary, exponential, fibonacci
        jump_cost = num_queries * math.sqrt(n)
        ternary_cost = num_queries * (math.log2(n) / math.log2(3))
        binary_cost = num_queries * math.log2(n)
        exp_cost = num_queries * math.log2(n)
        fib_cost = num_queries * math.log2(n)

        costs = {
            'jump_search': jump_cost,
            'ternary_search': ternary_cost,
            'binary_search': binary_cost,
            'exponential_search': exp_cost,
            'fibonacci_search': fib_cost
        }
        return min(costs, key=costs.get)


In [8]:
# Cell 4: Interactive prediction
def interactive_predict():
    try:
        print("Enter an array of numbers (comma separated):")
        user_input = input("Array: ").strip()
        user_input = user_input.strip('[]')
        arr = [float(x.strip()) for x in user_input.split(',') if x.strip()]

        if len(arr) == 0:
            print("Error: Array cannot be empty!")
            return

        print("Enter number of queries (k):")
        k = int(input().strip())

        print("\nCalculating features and predicting...")
        print(f"Sortedness (search): {calculate_sortedness_searching(arr):.4f}")
        print(f"Sorted : {calculate_sortedness_sorting(arr):.4f}")
        print(f"Uniqueness: {calculate_uniqueness(arr):.4f}")
        print(f"Range: {calculate_range(arr):.4f}")

        sort_algo = predict_sorting_algorithm(arr)
        print(f"\nPredicted Sorting Algorithm: {sort_algo}")

        search_algo = select_search_algorithm(arr, k)
        print(f"Selected Search Algorithm: {search_algo}")

    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    interactive_predict()


Enter an array of numbers (comma separated):
Enter number of queries (k):

Calculating features and predicting...
Sortedness (search): 1.0000
Sorted : 1.0000
Uniqueness: 1.0000
Range: 9.0000

Predicted Sorting Algorithm: Insertion
Selected Search Algorithm: interpolation_search
